# Text Feature Engineering — TF-IDF + LSA for Complaint-Topic Data

**Objective:** turn raw, unlabeled complaint text into a numeric feature matrix that `../code/kmeans_algorithms.ipynb` can load and cluster.
**Method:** TF-IDF (weighted term frequencies) -> Truncated SVD / LSA (dimensionality reduction).

**Proxy dataset:** [20 Newsgroups](https://scikit-learn.org/stable/datasets/real_world.html#the-20-newsgroups-text-dataset), 7 categories reframed as complaint topics — PC Hardware, Mac Hardware, Automotive, Motorcycle/Vehicle, Health/Medical Product, Electronics Product, Billing/Marketplace. The true category is kept in the saved output only to let the clustering notebook validate against it later.

In [1]:
import numpy as np
import pandas as pd
from sklearn.datasets import fetch_20newsgroups
from sklearn.feature_extraction.text import TfidfVectorizer, ENGLISH_STOP_WORDS
from sklearn.decomposition import TruncatedSVD
from sklearn.preprocessing import Normalizer
from sklearn.pipeline import make_pipeline

DATA_DIR = "."
RANDOM_STATE = 42

## STEP 01 - Load Raw Text

Headers/footers/quotes are stripped so the model works from the complaint body only, not metadata that would leak the category. Near-empty posts (pure quotes) are dropped.

In [2]:
CATEGORY_LABELS = {
    "comp.sys.ibm.pc.hardware": "PC Hardware",
    "comp.sys.mac.hardware":    "Mac Hardware",
    "rec.autos":                "Automotive",
    "rec.motorcycles":          "Motorcycle/Vehicle",
    "sci.med":                  "Health/Medical Product",
    "sci.electronics":          "Electronics Product",
    "misc.forsale":             "Billing/Marketplace",
}
categories = list(CATEGORY_LABELS.keys())

newsgroups = fetch_20newsgroups(
    subset="all",
    categories=categories,
    remove=("headers", "footers", "quotes"),
    random_state=RANDOM_STATE,
)

docs = newsgroups.data
true_labels = np.array(newsgroups.target)
true_label_names = [CATEGORY_LABELS[categories[i]] for i in true_labels]

print(f"Documents fetched: {len(docs)}")
pd.Series(true_label_names).value_counts()

Documents fetched: 6880


Health/Medical Product    996
Billing/Marketplace       990
Motorcycle/Vehicle        990
Electronics Product       984
PC Hardware               982
Automotive                975
Mac Hardware              963
Name: count, dtype: int64

In [3]:
mask = [len(d.strip()) > 20 for d in docs]
docs = [d for d, m in zip(docs, mask) if m]
true_labels = true_labels[mask]
true_label_names = [n for n, m in zip(true_label_names, mask) if m]
print(f"Documents after removing near-empty posts: {len(docs)}")

Documents after removing near-empty posts: 6630


## STEP 02 - Snapshot Raw Text

`fetch_20newsgroups` caches to `~/scikit_learn_data`, outside this repo (extracting ~20,000 files inside a OneDrive-synced folder is unreliable) — so the filtered subset actually used is saved here instead, for reproducibility.

In [4]:
snapshot = pd.DataFrame({"text": docs, "category": true_label_names})
snapshot.to_parquet(f"{DATA_DIR}/complaint_topics_subset.parquet", index=False)
print(f"Snapshot saved: {len(snapshot)} documents -> {DATA_DIR}/complaint_topics_subset.parquet")

Snapshot saved: 6630 documents -> ./complaint_topics_subset.parquet


## STEP 03 - TF-IDF

Stopwords are the standard English list plus a short extension (informal fillers like "just", "think", "know") added after an earlier pass showed them dominating cluster centroids downstream instead of real topic words.

In [5]:
FILLER_WORDS = [
    "just", "like", "don", "ve", "does", "know", "think", "good",
    "did", "say", "said", "got", "going", "make", "want", "really",
]
stop_words = list(ENGLISH_STOP_WORDS) + FILLER_WORDS

vectorizer = TfidfVectorizer(
    stop_words=stop_words,
    max_df=0.5,
    min_df=5,
    max_features=20000,
    sublinear_tf=True,
)
X_tfidf = vectorizer.fit_transform(docs)
print(f"TF-IDF matrix: {X_tfidf.shape[0]} documents x {X_tfidf.shape[1]} terms")

TF-IDF matrix: 6630 documents x 9387 terms


## STEP 04 - Dimensionality Reduction (Truncated SVD / LSA)

PCA needs dense, mean-centered data; Truncated SVD does the equivalent reduction directly on the sparse TF-IDF matrix. Output is L2-normalized so downstream Euclidean distance behaves like cosine similarity.

In [6]:
svd = TruncatedSVD(n_components=100, random_state=RANDOM_STATE)
normalizer = Normalizer(copy=False)
lsa = make_pipeline(svd, normalizer)

X_lsa = lsa.fit_transform(X_tfidf)
explained = svd.explained_variance_ratio_.sum()
print(f"Reduced {X_tfidf.shape[1]} terms -> {X_lsa.shape[1]} LSA dimensions, "
      f"explaining {explained:.1%} of the TF-IDF variance")

Reduced 9387 terms -> 100 LSA dimensions, explaining 13.3% of the TF-IDF variance


## STEP 05 - Save Feature Matrix

Saved as plain Parquet, not a pickled sklearn object — portable, and it's all the clustering notebook needs.

In [7]:
features_df = pd.DataFrame(X_lsa, columns=[f"feature_{i}" for i in range(X_lsa.shape[1])])
features_df["true_label"] = true_labels
features_df["true_label_name"] = true_label_names
features_df.to_parquet(f"{DATA_DIR}/complaint_features.parquet", index=False)
print(f"Feature matrix saved -> {DATA_DIR}/complaint_features.parquet  {features_df.shape}")

Feature matrix saved -> ./complaint_features.parquet  (6630, 102)
